# 04 Gravity Field API

In [1]:
from pathlib import Path
import sys

_candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve().parent.parent]
_PROJECT_ROOT = next((p for p in _candidates if (p / 'src' / 'solsys_emulator').exists()), None)
if _PROJECT_ROOT is None:
    raise RuntimeError('Impossibile trovare la project root con src/solsys_emulator')

_SRC = _PROJECT_ROOT / 'src'
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

print('Project root:', _PROJECT_ROOT)
print('Python executable:', sys.executable)



Project root: /Users/francescosulli/Documents/ASTREO/IA e simulazioni/solar_system
Python executable: /Applications/Xcode.app/Contents/Developer/usr/bin/python3


In [3]:
import numpy as np

from solsys_emulator.config import DEFAULT_CHECKPOINT_PATH
from solsys_emulator.constants import get_mu
from solsys_emulator.gravity_field import acceleration, potential
from solsys_emulator.inference import EphemerisEmulator

emu = EphemerisEmulator.from_checkpoint(DEFAULT_CHECKPOINT_PATH)
t_ref = '2025-02-15T00:00:00'
state = emu.predict_state(t_ref)

# Punto vicino alla Terra: +10,000 km lungo asse x locale.
earth_r = state['earth']['r'].to_value('km')
point = earth_r + np.array([10_000.0, 0.0, 0.0])

states_positions = {body: {'r': body_state['r']} for body, body_state in state.items()}
mu_values = {body: get_mu(body) for body in state.keys()}

a_total = acceleration(point, states_positions, mu_values, epsilon=0.0)
phi_total = potential(point, states_positions, mu_values, epsilon=0.0)

print('a_total [km/s^2]:', a_total)
print('|a_total| [km/s^2]:', np.linalg.norm(a_total))
print('phi_total [km^2/s^2]:', phi_total)

# Mini sanity check: contributo Terra dominante ~ mu/r^2 su 10,000 km.
mu_earth = get_mu('earth')
r_test = 10_000.0
a_expected = mu_earth / (r_test**2)
print('Earth-only magnitude mu/r^2 [km/s^2]:', a_expected)


a_total [km/s^2]: [-3.98096501e-03 -3.09999222e-06 -1.34368767e-06]
|a_total| [km/s^2]: 0.003980966444914291
phi_total [km^2/s^2]: -938.6002355010221
Earth-only magnitude mu/r^2 [km/s^2]: 0.00398600435507
